In [1]:
!pip install pandas openpyxl numpy

# Lihat Preview Pilot Testing

In [5]:
import pandas as pd

nama_file_csv = 'pilot_testing_enriched.csv'

try:
    df = pd.read_csv(nama_file_csv)
    
    display(df.tail())

except FileNotFoundError:
    print(f"Error: File '{nama_file_csv}' tidak ditemukan.")
    print("Pastikan file CSV sudah di-upload ke folder yang sama dengan notebook ini.")

,ad_id,title,price,lat,lon,Provinsi,Kota/kabupaten,Kecamatan,Luas_bangunan,Luas_tanah,...,Fasilitas,Lantai,description,Fasilitas_Indoor,Karakteristik_bangunan,Kelurahan/desa,Kamar_Mandi_Tamu,Sertifikat,Daya_Listrik (Watt),Arah_Hadap
55,941748797,Rumah Siap Huni 2 Lantai Pantai Mentari Suraba...,3.000000e+09,-7.240,112.793,Jawa Timur,Surabaya Kota,Bulak,300,250,...,"garasi, garden",2.0,DIJUAL RUMAH 2 LANTAI PANTAI MENTARI LT: ...,"Garasi, Tanpa perabotan, Halaman terbuka","Taman, Lapangan tenis",Kenjeran,NaN,SHM,NaN,Utara
56,918081428,RUMAH BARU MINIMALIS DUA LANTAI SIAP HUNI MEDO...,1.500000e+09,-7.278,112.755,Jawa Timur,Surabaya Kota,Gubeng,115,70,...,"ac, garasi, garden",2.0,Dijual Rumah BARU Minimalis 2 lt Medokan Asri...,"Garasi, Halaman terbuka, Pendingin ruangan (AC...","Keamanan, Taman",Airlangga,NaN,SHM,NaN,Selatan
57,941750311,Dijual Apartemen Educity Pakuwon City,2.750000e+08,-7.279,112.806,Jawa Timur,Surabaya Kota,Mulyorejo,22,0,...,NaN,16.0,"Dijual Apartemen Educity, Pakuwon City Tower ...",Berperabot lengkap,NaN,Kejawen Putih Tambak,NaN,NaN,NaN,NaN
58,940008663,Dijual rumah laguna Pakuwon model american cla...,2.150000e+09,-7.279,112.802,Jawa Timur,Surabaya Kota,Mulyorejo,120,90,...,"ac, garasi, garden",2.0,Dijual Rumah bagus dan cluster premium berloka...,"Garasi, Pendingin ruangan (AC), Kitchen set, D...","Keamanan, Gym, Angkat, Pustaka, Pramutamu, BBQ...",Kejawen Putih Tambak,3.0,SHM,7700,NaN
59,929539604,rangka sby rumah progres minimalis,3.100000e+08,-7.253,112.756,Jawa Timur,Surabaya Kota,Tambaksari,40,20,...,NaN,1.0,Rumah minimalis rangka sby UK 4x5m 2kt.2km Le...,Tanpa perabotan,NaN,Tambaksari,NaN,SHM,NaN,NaN


# Konversi ke Excel

In [6]:
nama_file_excel = 'pilot_testing_enriched.xlsx'

try:
    df.to_excel(nama_file_excel, index=False, engine='openpyxl')
    
except Exception as e:
    print(f"Terjadi kesalahan saat menyimpan file: {e}")

# Standardisasi Istilah Fasilitas

In [ ]:
synonym_map = {
    "garden": "Taman",
    "garasi": "Parkiran mobil",
    "carport": "Parkiran mobil",
    "swimmingpool": "Kolam renang",
    "gordyn": "Gorden",
    "pam": "Air",
    "waterheater": "Pemanas air",
    "refrigerator": "Kulkas",
    "stove": "Kompor",
    "fireextenguisher": "APAR (Alat Pemadam Api Ringan)",
    "ac": "Pendingin ruangan (AC)",
    "pendingin ruangan (ac)": "Pendingin ruangan (AC)",
    "keamanan": "Sistem Keamanan",
    "keamanan 24 jam": "Sistem Keamanan",
    "telephone": "Telepon"
}

def process_facilities(row):
    # Gabungkan kolom "Fasilitas", "Fasilitas_Indoor", dan "Karakteristik_bangunan"
    f1 = str(row['Fasilitas']) if pd.notna(row['Fasilitas']) else ""
    f2 = str(row['Fasilitas_Indoor']) if pd.notna(row['Fasilitas_Indoor']) else ""
    f3 = str(row['Karakteristik_bangunan']) if pd.notna(row['Karakteristik_bangunan']) else ""
    
    raw_combined = f"{f1},{f2},{f3}"
    items = [item.strip().lower() for item in raw_combined.split(',')]
    
    standardized_items = set()
    for item in items:
        if item == "" or item == "nan":
            continue
        mapped_value = synonym_map.get(item, item.title())
        standardized_items.add(mapped_value)
        
    return ", ".join(sorted(standardized_items))

# Eksekusi Standardisasi

In [16]:
df['Facilities'] = df.apply(process_facilities, axis=1)

df = df.drop(columns=['Fasilitas', 'Fasilitas_Indoor', 'Karakteristik_bangunan'])

df[['title', 'Facilities']].head(5)

,title,Facilities
0,"Rumah Baru 2 Lantai, Bagus dan Strategis di Wi...","Air, Listrik, Pagar Penuh, Parkiran mobil, Tan..."
1,Lantai 2‼️155 jt • 2 BR Termurah Apartemen Pun...,Sebagian Perabotan
2,Diamond Hill Citraland,"Parkiran mobil, Tanpa Perabotan"
3,"‼️BARU GRESS 2 UNIT‼️ RUMAH WISMA MUKTI, SEMAL...",Tanpa Perabotan
4,RUMAH MEWAH SIAP HUNI 2 LT DHARMAHUSADA SURABA...,"Listrik, Tanpa Perabotan"


# Simpan Update Pilot Testing

In [17]:
df.to_csv("pilot_testing_cleaned.csv", index=False)